In [ ]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

In [ ]:
location = "india"
vehicle = "rice"
scenario = "intervention"

In [ ]:
def aggregate_by_scenario(df):
    return df.groupby(["scenario", "input_draw", "wealth_quintile"]).value.sum().groupby(["scenario", "wealth_quintile"]).mean()

In [ ]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = (
        pd.read_parquet(path)
    )
else:
    pregnancy_person_time_anemia = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )
pregnancy_person_time_anemia

In [ ]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

In [ ]:
pregnancy_person_time_anemia.sub_entity.value_counts()

In [ ]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

In [ ]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[pregnancy_person_time_anemia.sub_entity != 'not_anemic']
)
anemic_pregnant_person_time

In [ ]:
pregnant_anemia_prevalence_by_scenario = (anemic_pregnant_person_time / total_pregnant_person_time).fillna(0)
pregnant_anemia_prevalence_by_scenario

In [ ]:
path = f'./results/{location}/{vehicle}/{scenario}/pregnant_anemia_prevalence_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [ ]:
pop = pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
pop

In [ ]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

In [ ]:
pregnancy_prevalent_anemia_cases_by_scenario = pregnant_anemia_prevalence_by_scenario * pregnant_pop
pregnancy_prevalent_anemia_cases_by_scenario

In [ ]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = (
        pd.read_parquet(path)
    )
else:
    maternal_disorders_transition_counts = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

maternal_disorders_transition_counts

In [ ]:
maternal_disorders_transition_counts.sub_entity.cat.categories

In [ ]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[maternal_disorders_transition_counts.sub_entity == 'susceptible_to_maternal_disorders_to_maternal_disorders']
)
maternal_disorders_incident_cases_by_scenario

In [ ]:
path = f'./results/{location}/{vehicle}/{scenario}/maternal_disorders_incident_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [ ]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
    )
else:
    neonatal_deaths = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet").assign(value=0).assign(maternal_scenario=lambda x: x.maternal_scenario.replace('intervention', scenario))
    )

neonatal_deaths = neonatal_deaths.rename(columns={"maternal_scenario": "scenario"})
neonatal_deaths

In [ ]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

In [ ]:
path = f'./results/{location}/{vehicle}/{scenario}/neonatal_deaths_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [ ]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = (
        pd.read_parquet(path)
    )
else:
    non_pregnancy_anemia_cases = (
        pd.read_parquet(f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/anemia_cases.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

non_pregnancy_anemia_cases

In [ ]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0"))
non_pregnancy_prevalent_anemia_cases_by_scenario

In [ ]:
prevalent_anemia_cases_by_scenario = pregnancy_prevalent_anemia_cases_by_scenario + non_pregnancy_prevalent_anemia_cases_by_scenario
prevalent_anemia_cases_by_scenario

In [ ]:
path = f'./results/{location}/{vehicle}/{scenario}/prevalent_anemia_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [ ]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = (
        pd.read_csv(path)
    )
else:
    ntd_cases_by_scenario = (
        pd.read_csv(f"../0500_neural_tube_defects_model/results/india/rice/intervention/ntd_cases_by_scenario.csv").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(["scenario", "wealth_quintile"]).value
ntd_cases_by_scenario

In [ ]:
path = f'./results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)